To get the path of data.

In [1]:
import os
print(os.getcwd())


/app


In [2]:
import os

print(os.listdir("D:\projects\restaurant_data_pipeline\data\data1"))


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\projects\restaurant_data_pipeline\\data\\data1'

In [ ]:
from pyspark.sql import SparkSession

# إنشاء SparkSession
spark = SparkSession.builder.appName("ReadSingleJSON").getOrCreate()

# تحديد المسار الكامل للملف المحدد
json_file_path = "/home/jovyan/work/data/data1"



# قراءة الملف فقط
df = spark.read.json(json_file_path)

# عرض البيانات
df.show()
df.printSchema()


In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)


In [ ]:
from pyspark.sql.functions import col, explode_outer
from pyspark.sql.types import StructType, ArrayType

def fully_flatten(df, prefix=""):
    while True:
        complex_fields = [
            (field.name, field.dataType)
            for field in df.schema.fields
            if isinstance(field.dataType, (StructType, ArrayType))
        ]
        if not complex_fields:
            break

        for field_name, data_type in complex_fields:
            new_prefix = f"{prefix}{field_name}_" if prefix else f"{field_name}_"
            if isinstance(data_type, StructType):
                subfields = [
                    col(f"{field_name}.{subfield.name}").alias(f"{new_prefix}{subfield.name}")
                    for subfield in data_type.fields
                ]
                df = df.select(
                    *[col(c) for c in df.columns if c != field_name],
                    *subfields
                )
            elif isinstance(data_type, ArrayType):
                df = df.withColumn(field_name, explode_outer(col(field_name)))
    return df

# 1. انفجر restaurants فقط
df_restaurants = df.withColumn("restaurant", explode_outer("restaurants")).select("restaurant.*")

# 2. فك كل التداخلات لأي Struct أو Array جوا restaurant فقط
df_fully_flat = fully_flatten(df_restaurants, prefix="restaurant_")


df_fully_flat.printSchema()


In [ ]:
df_fully_flat.limit(10)

In [ ]:
print("عدد الصفوف الأصلي:", df_fully_flat.count())


In [ ]:

print("column names:", df_fully_flat.columns)




In [ ]:
import re

def clean_column_name(col_name):
    # يشيل كل تكرار لعبارة restaurant_ في بداية الاسم
    return re.sub(r'^(restaurant_)+', '', col_name)

# إعادة تسمية الأعمدة في DataFrame
for old_name in df_fully_flat.columns:
    new_name = clean_column_name(old_name)
    if new_name != old_name:
        df_fully_flat = df_fully_flat.withColumnRenamed(old_name, new_name)

print("column names:", df_fully_flat.columns)


In [ ]:
from pyspark import StorageLevel

# فحص شامل للبيانات
print("=== information about the data ===")
print(f"TOtal of records: {df_fully_flat.count():,}")
print(f"Total of raws: {len(df_fully_flat.columns)}")

# إزالة التكرارات مع persist
df_deduped = df_fully_flat.dropDuplicates().persist(StorageLevel.MEMORY_AND_DISK)

print("=== information about the data ===")
print(f"TOtal of records: {df_deduped.count():,}")  # سريع!
print(f"Total of raws: {len(df_deduped.columns)}")

# تنظيف الذاكرة عند الانتهاء
df_deduped.unpersist()


In [ ]:
final_result= df_deduped
final_result.limit(20)

In [ ]:
from google.cloud import bigquery
import os

# تحديد مسار ملف الخدمة
key_path = "/home/jovyan/keys/fooddelivary-456823-44ced47a2164.json"

# إنشاء عميل BigQuery
client = bigquery.Client.from_service_account_json(key_path)

# اسم الجدول النهائي
table_id = "fooddelivary-456823.food_analytics.resturant"

# 1. التأكد إذا كان الجدول موجود
try:
    client.get_table(table_id)
    print("✅ الجدول موجود وسيتم حذفه أولاً...")
    client.delete_table(table_id)
    print("✅ تم حذف الجدول القديم.")
except Exception as e:
    print("ℹ️ الجدول غير موجود وسيتم إنشاؤه جديد.")

# 2. رفع الداتا فريم إلى BigQuery (سينشئ الجدول تلقائيًا)
job = client.load_table_from_dataframe(final_result.toPandas(), table_id).result()

print("✅ تم رفع final_result إلى BigQuery بنجاح باسم resturant.")
